# Chapter 9 — Convolutions

**Math for ML Engineers** · companion notebook

This notebook mirrors §9.2–§9.4 of Chapter 9 (convolution as structured matrix multiply, the 2-D forward/backward pass, and receptive field growth), the mathematical core that every CNN — from LeNet to ResNet — is built on. **Key takeaway: a convolution is just a matrix multiply by a sparse, weight-shared (Toeplitz) matrix, and its gradients follow directly from that fact — backprop through a conv layer is itself a convolution.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## §9.2 — Convolution as a structured matrix multiply

For a 1-D "valid" convolution of signal $x \in \mathbb{R}^n$ with kernel $w \in \mathbb{R}^k$, the output

$$
y_i = \sum_{j=0}^{k-1} w_j\, x_{i+j}, \qquad i = 0, \dots, n-k
$$

is *linear* in $x$, so it can be written exactly as $y = C_w x$ for some matrix $C_w \in \mathbb{R}^{(n-k+1) \times n}$. Each row of $C_w$ is the kernel $w$ shifted one step to the right of the previous row — a **Toeplitz matrix** (constant along diagonals). This is the same object that appears in the "doubly block-Toeplitz" construction used for 2-D convolutions: stack block-Toeplitz blocks that are themselves Toeplitz.

Once we see $y = C_w x$, everything from Chapters 7–8 (linear layers, their Jacobians, backprop through matrix multiply) applies unchanged to convolutions — a conv layer *is* a linear layer with a highly structured, weight-tied matrix.

In [ ]:
def conv1d_toeplitz(x, w):
    """1-D valid convolution via explicit Toeplitz matrix. Illustrates conv IS matrix multiply: y = C_w x"""
    n, k = len(x), len(w)
    out_len = n - k + 1
    C = np.zeros((out_len, n))
    for i in range(out_len):
        C[i, i:i+k] = w
    return C @ x, C


def conv1d(x, w):
    k = len(w)
    return np.array([np.dot(x[i:i+k], w) for i in range(len(x) - k + 1)])


x_demo = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
w_demo = np.array([1.0, 0.0, -1.0])  # simple 1-D edge/derivative kernel

y_toeplitz, C_w = conv1d_toeplitz(x_demo, w_demo)
y_direct = conv1d(x_demo, w_demo)

print("y via Toeplitz matmul:", y_toeplitz)
print("y via direct sliding window:", y_direct)
print("Match:", np.allclose(y_toeplitz, y_direct))
print("C_w shape:", C_w.shape)


## §9.3 — 2-D convolution, forward and backward

The 2-D case extends the same idea over two axes. For "valid" (no padding) 2-D convolution with $x \in \mathbb{R}^{H \times W}$ and kernel $w \in \mathbb{R}^{k_H \times k_W}$:

$$
y_{i,j} = \sum_{u=0}^{k_H-1}\sum_{v=0}^{k_W-1} w_{u,v}\, x_{i+u,\, j+v}
$$

Because this is linear in both $x$ and $w$, backprop gives closed-form conv-shaped gradients:

$$
\frac{\partial L}{\partial x} = \text{full\_conv}\!\left(\frac{\partial L}{\partial y},\, \text{flip}(w)\right), \qquad
\frac{\partial L}{\partial w} = \text{valid\_conv}\!\left(x,\, \frac{\partial L}{\partial y}\right)
$$

Intuitively: the gradient w.r.t. the *input* is a **transposed convolution** — the upstream gradient $dy$ is zero-padded and convolved with the *spatially flipped* kernel, spreading each output gradient back over the receptive field it came from. The gradient w.r.t. the *kernel* is a valid convolution of the input with the upstream gradient — each weight's gradient sums contributions from every spatial position it touched (weight tying → gradients accumulate).

In [ ]:
def conv2d_valid(x, w):
    """2-D valid convolution (no padding). x:(H,W) w:(kH,kW) -> y:(H-kH+1, W-kW+1)"""
    kH, kW = w.shape
    oH = x.shape[0] - kH + 1
    oW = x.shape[1] - kW + 1
    y  = np.zeros((oH, oW))
    for i in range(oH):
        for j in range(oW):
            y[i, j] = np.sum(x[i:i+kH, j:j+kW] * w)
    return y


def conv2d_backward(x, w, dy):
    """dx = full_conv(dy, flip(w)) [backprop through conv = transposed conv]; dw = valid_conv(x, dy)"""
    kH, kW = w.shape
    pH, pW = kH - 1, kW - 1
    dy_pad = np.pad(dy, ((pH, pH), (pW, pW)))
    w_flip = w[::-1, ::-1]
    dx = conv2d_valid(dy_pad, w_flip)
    dw = conv2d_valid(x, dy)
    return dx, dw


def receptive_field(n_layers, kernel_size, stride=1):
    """RF_L = 1 + L*(k-1) for stride=1. rf = stride*(rf-1) + kernel_size iterated."""
    rf = 1
    for _ in range(n_layers):
        rf = stride * (rf - 1) + kernel_size
    return rf


# Gradient check: finite differences vs analytic conv2d_backward
np.random.seed(0)
x_gc = np.random.randn(6, 6)
w_gc = np.random.randn(3, 3)
y_gc = conv2d_valid(x_gc, w_gc)
dy_gc = np.random.randn(*y_gc.shape)

dx_analytic, dw_analytic = conv2d_backward(x_gc, w_gc, dy_gc)

eps = 1e-6
dw_numeric = np.zeros_like(w_gc)
for u in range(w_gc.shape[0]):
    for v in range(w_gc.shape[1]):
        w_plus = w_gc.copy();  w_plus[u, v]  += eps
        w_minus = w_gc.copy(); w_minus[u, v] -= eps
        loss_plus  = np.sum(conv2d_valid(x_gc, w_plus)  * dy_gc)
        loss_minus = np.sum(conv2d_valid(x_gc, w_minus) * dy_gc)
        dw_numeric[u, v] = (loss_plus - loss_minus) / (2 * eps)

print("max |dw_analytic - dw_numeric| =", np.max(np.abs(dw_analytic - dw_numeric)))
print("Gradient check passed:", np.allclose(dw_analytic, dw_numeric, atol=1e-4))


## Visualizing the structure

Three plots make the algebra above concrete:

1. **The Toeplitz matrix $C_w$ itself**, as a heatmap — the diagonal band of repeated kernel weights is the visual signature of "convolution is a sparse, weight-shared matrix multiply."
2. **A Sobel-like edge kernel applied to a synthetic block-checkerboard image** (4×4-pixel blocks, so the 3×3 kernel actually straddles real edges instead of single-pixel alternation) via `conv2d_valid` — input vs. output side by side, showing that a fixed 3×3 kernel is literally sweeping the matrix-multiply view from panel 1 across two spatial axes.
3. **Receptive field vs. depth** for kernel sizes 3 and 5 at stride 1 — using `receptive_field` to show how many layers it takes before a single output unit "sees" the whole input.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- Panel 1: Toeplitz matrix heatmap ---
im0 = axes[0].imshow(C_w, cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
axes[0].set_title(r"$C_w$: Toeplitz matrix for $w=[1,0,-1]$")
axes[0].set_xlabel("input index j")
axes[0].set_ylabel("output index i")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# --- Panel 2: Sobel-like edge detection on a block checkerboard ---
size, block = 12, 4  # block > 1 so 3x3 Sobel sees real edges, not single-pixel noise
xs, ys = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
checker = (((xs // block) + (ys // block)) % 2).astype(float)
sobel_x = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]], dtype=float)
edges = conv2d_valid(checker, sobel_x)

axes[1].imshow(checker, cmap="gray")
axes[1].set_title("Input: checkerboard")
axes[1].axis("off")

# --- Panel 3 needs its own subplot; redo layout for edge output + RF plot ---
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(11, 4.5))
axes2[0].imshow(edges, cmap="gray")
axes2[0].set_title("Sobel-x output: conv2d_valid(checker, sobel_x)")
axes2[0].axis("off")

layers = np.arange(1, 21)
rf_k3 = [receptive_field(int(L), 3) for L in layers]
rf_k5 = [receptive_field(int(L), 5) for L in layers]
axes2[1].plot(layers, rf_k3, marker="o", label="kernel_size=3")
axes2[1].plot(layers, rf_k5, marker="s", label="kernel_size=5")
axes2[1].set_xlabel("number of layers (depth)")
axes2[1].set_ylabel("receptive field (pixels)")
axes2[1].set_title("Receptive field vs. depth (stride=1)")
axes2[1].legend()
axes2[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("C_w nonzero fraction:", np.count_nonzero(C_w) / C_w.size, "(sparse!)")
print("Edge map shape:", edges.shape, " max |edge| response:", np.max(np.abs(edges)))
print("RF after 10 layers, k=3:", receptive_field(10, 3), " k=5:", receptive_field(10, 5))


## Why this matters in production: the receptive-field-per-layer tradeoff

The plot above shows $\text{RF}_L = 1 + L(k-1)$ for stride 1 — **receptive field grows only *linearly* with depth**. This single fact shaped a decade of CNN architecture design: to let a network "see" a large fraction of a 224×224 ImageNet image with 3×3 kernels, you either need dozens of layers (VGG-19, ResNet-50/101/152 all exist partly *because* linear RF growth forces depth) or you need cheaper tricks — strided convolutions and pooling to downsample (each stride-2 layer multiplies the *effective* input-pixel RF, not just adds to it), or dilated/atrous convolutions (as in WaveNet and DeepLab) that skip input pixels to grow RF exponentially without adding parameters or true depth.

This tradeoff has a direct analog in why **self-attention displaced convolutions for sequence modeling**. In a stride-1 CNN, connecting two positions $n$ tokens apart requires $O(n)$ layers (linear RF growth, exactly as plotted above). In a self-attention layer, every token attends directly to every other token — the "receptive field" is the *entire sequence* after a **single layer**, i.e. $O(1)$ layers vs. $O(n)$ layers for equivalent context. That's not the only reason Transformers overtook CNNs/RNNs for text (parallelizability across the sequence dimension matters at least as much), but the receptive-field-per-layer asymptotics are a core, textbook-cited reason self-attention was worth its $O(n^2)$ per-layer cost.

In [ ]:
def layers_to_cover(seq_len, kernel_size, stride=1):
    """How many stacked conv layers (stride=1) are needed until RF >= seq_len?"""
    n_layers = 0
    while receptive_field(n_layers, kernel_size, stride) < seq_len:
        n_layers += 1
    return n_layers


seq_lengths = [16, 64, 256, 1024, 4096]
print(f"{'seq_len':>8} | {'layers (k=3)':>13} | {'layers (k=5)':>13} | {'attn layers':>12}")
print("-" * 56)
for L in seq_lengths:
    layers_k3 = layers_to_cover(L, 3)
    layers_k5 = layers_to_cover(L, 5)
    attn_layers = 1  # full-sequence RF in a single self-attention layer, any L
    print(f"{L:>8} | {layers_k3:>13} | {layers_k5:>13} | {attn_layers:>12}")

print()
print("A stride-1 CNN needs O(seq_len) layers to reach full-context receptive field;")
print("self-attention reaches it in O(1) layer, regardless of sequence length.")


## Exercise

1. `conv1d_toeplitz` builds $C_w$ for *valid* (no padding) 1-D convolution. Derive and implement the Toeplitz matrix for *same* padding (output length equals input length) and for *full* convolution (output length $n+k-1$), and verify each against `np.convolve(x, w, mode=...)`. Which of the two padding regimes keeps $C_w$ square, and why does that matter for the transposed-convolution view used in `conv2d_backward`? (See §9.2 on the convolution backward pass and its transposed-matrix view.)
2. Using `receptive_field`, derive a closed-form expression for the number of stride-1 layers $L^\*$ needed to reach a target receptive field $R$ for a given kernel size $k$, then extend `receptive_field` to accept a *list* of per-layer strides (rather than one shared stride) and show that a single stride-2 layer inserted early can cut $L^\*$ roughly in half for large $R$. Relate this result to the effective-context-per-layer comparison against self-attention worked out above. (See §11.9 on attention's asymptotic complexity.)